# Building a first Classification NN

Use the Ames Mutagenicity dataset (from assignment 1A) and build a binary classifier NN. Play with the model parameters. 

For comparison of the NN model performance, consider the performance of other (baseline) classifier models (assignment 1A):
- KNN: Test-Accuracy 0.79, Test-ROC-AUC 0.86
- Decision Tree: Test-Accuracy 0.78, Test-ROC-AUC 0.77
- Random Forest: Test-Accuracy 0.83, Test-ROC-AUC 0.90
- Gradient Boosting: Test-Accuracy 0.77, Test-ROC-AUC 0.85


#### Tasks:
1) Load the dataset `ames_data.csv`. The dataset does not contain any duplicates or NaNs
2) Feature engineering: Calculate various fingerprints from the SMILES strings via mol objects using RDKit(snippet provided for Morgan FPs and MACCS keys)
3) Create feature matrix and target vector. Choose first the MorganFP (Later repeat the process for other fingerprint types). Convert the training and test sets into pytorch tensors.
4) Build your NN (see below for more info)
5) Train your model on the Morgan Fingerprints (and repeat later for other FP types)
6) Evaluate your model's performance and compare to other classifier models
7) Save the model / current state.
8) Respond to the discussion points


#### Note:
The aim of this exercise is to gain a bit of practice in building a simple NN and to see how different parameters and feature engineering influence the model. Maximum accuracy is not the target. 

There is no need to venture too far into the details or more advanced approaches just yet (e.g. batched training would be complete overkill for this assignment - we will discuss that in the next sessions)

0) Import dependencies and datasets

In [1]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys
from rdkit.DataStructs import ConvertToNumpyArray

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# ==============================
# Imports
# ==============================
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys
from rdkit.DataStructs import ConvertToNumpyArray

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

import torch
import torch.nn as nn
import torch.optim as optim

1) Load and investigate the data

In [3]:
df = pd.read_csv("ames_data.csv")
df.head()

,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1


In [4]:
print(df.shape)
df.head()

(7278, 3)


,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1


2) Generate different fingerprints (try at least one additional FP type as provided in RDKit and use two different fpSizes on one of them) - all of them will be saved in new columns in the Dataframe.

In [5]:
def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol

def bitvect_to_np(fp):
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    ConvertToNumpyArray(fp, arr)
    return arr

def morganfp_1024(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024).GetFingerprint(mol)
    return bitvect_to_np(fp)

def morganfp_2048(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol)
    return bitvect_to_np(fp)

def maccskeys_fp(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    return bitvect_to_np(fp)

In [6]:
df["mol"] = df["smiles"].apply(smiles_to_mol)

# check valid molecules
print("Invalid mols:", df["mol"].isna().sum())

# generate fingerprints
df["MorganFP_1024"] = df["mol"].apply(morganfp_1024)
df["MorganFP_2048"] = df["mol"].apply(morganfp_2048)
df["MACCSkeys"] = df["mol"].apply(maccskeys_fp)

df.head()

Invalid mols: 0


,drug_id,smiles,mutagenicity,mol,MorganFP_1024,MorganFP_2048,MACCSkeys
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1,<rdkit.Chem.rdchem.Mol object at 0x00000239077...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1,<rdkit.Chem.rdchem.Mol object at 0x00000239077...,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0,<rdkit.Chem.rdchem.Mol object at 0x00000239077...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1,<rdkit.Chem.rdchem.Mol object at 0x00000239077...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1,<rdkit.Chem.rdchem.Mol object at 0x00000239077...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


3. Create feature matrix and target vector. The snippet below converts the data into numpy arrays. Start with the Morgan Fingerprints (and later return here to apply your modell to different fingerprint types - not all of the fingerprints may have the same length, so you may have to adapt the width of your layers).

Do a train startified test split and convert into pytorch tensors.

In [7]:
def prepare_data(df, fp_column, target_column="mutagenicity", test_size=0.2, random_state=42):
    X = np.stack(df[fp_column].values).astype(np.float32)
    y = df[target_column].to_numpy().astype(np.float32)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.float32)

    return X_train_t, X_test_t, y_train_t, y_test_t, X_train, X_test, y_train, y_test

4) Build the NN - adhere to some robust standard values. Start simple and train the model on Morgan FP first.

Optimise the model parameters based on observed over-/underfitting. Experiment with different width and depth, as well as other model parameters. Explore some options to prevent overfitting, e.g. Early stopping (e.g. manually by limiting the epochs) or dropouts. 

Note: Since the input layer needs a lot of neurons (e.g. 2048 bit in the MFPs), consider shrinking the widht from layer to layer. 

Hint: If you use `BCELoss()` as loss function, combine it with a `sigmoid` activation in the last layer. If you use `BCEWithLogitsLoss()`, do not specify any activation in the forward pass (`x = self.outputlayer(x)`).

In [8]:
class BinClassifierNN(nn.Module):
    def __init__(self, input_dim, hidden1=512, hidden2=128, dropout=0.3):
        super(BinClassifierNN, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden2, 1)
        )

    def forward(self, x):
        return self.net(x)

In [9]:
# Parameters (change and add as needed)
learning_rate = 0.01
num_epochs = 100

5. Train the NN. Note that you may have to squeeze the output (`outputs=models(X_train).squeeze`). This will reduce the actual output of the shape ``[N, 1]`` to ``[N]``, which is comparable to y (The final layer naturally produces a column tensor, which is not directly comparable to the 1D target tensor).

In [11]:
def train_and_evaluate(
    X_train_t, X_test_t, y_train_t, y_test_t,
    input_dim,
    learning_rate=0.001,
    num_epochs=100,
    hidden1=512,
    hidden2=128,
    dropout=0.3,
    patience=15
):
    model = BinClassifierNN(
        input_dim=input_dim,
        hidden1=hidden1,
        hidden2=hidden2,
        dropout=dropout
    )

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_test_loss = float("inf")
    best_state = None
    patience_counter = 0

    train_losses = []
    test_losses = []

    for epoch in range(num_epochs):
        # ---- training ----
        model.train()
        outputs = model(X_train_t).squeeze()
        loss = criterion(outputs, y_train_t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # ---- evaluation loss on test ----
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test_t).squeeze()
            test_loss = criterion(test_outputs, y_test_t)

        train_losses.append(loss.item())
        test_losses.append(test_loss.item())

        # early stopping
        if test_loss.item() < best_test_loss:
            best_test_loss = test_loss.item()
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {loss.item():.4f} | Test Loss: {test_loss.item():.4f}")

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # load best model
    if best_state is not None:
        model.load_state_dict(best_state)

    # final evaluation
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t).squeeze()
        probs = torch.sigmoid(logits).numpy()
        preds = (probs >= 0.5).astype(int)

    y_true = y_test_t.numpy().astype(int)

    acc = accuracy_score(y_true, preds)
    roc = roc_auc_score(y_true, probs)

    print("\nFinal Test Performance")
    print("Accuracy:", round(acc, 4))
    print("ROC-AUC :", round(roc, 4))
    print("\nClassification Report:")
    print(classification_report(y_true, preds))

    return model, acc, roc, train_losses, test_losses

6) Evaluate the model. As a first metric, you can use the same loss function to evaluate the model on the test set. For better comparison with methods tested in the assignment 1A (results vide supra), use metrics from scikit-learn (e.g. the accuracy or ROC-AUC score).

Hint: for your prediction you may have to use `squeeze` again to match the target vector in the test set (e.g. ``y_pred = model(X_test).squeeze()``)

In [13]:
X_train_t, X_test_t, y_train_t, y_test_t, X_train, X_test, y_train, y_test = prepare_data(
    df, fp_column="MorganFP_2048"
)

model_morgan2048, acc_morgan2048, roc_morgan2048, train_losses, test_losses = train_and_evaluate(
    X_train_t, X_test_t, y_train_t, y_test_t,
    input_dim=X_train.shape[1],
    learning_rate=0.001,
    num_epochs=100,
    hidden1=512,
    hidden2=128,
    dropout=0.3,
    patience=15
)

Epoch [10/100] | Train Loss: 0.5304 | Test Loss: 0.5388
Epoch [20/100] | Train Loss: 0.3378 | Test Loss: 0.4961
Epoch [30/100] | Train Loss: 0.2303 | Test Loss: 0.5518
Early stopping at epoch 32

Final Test Performance
Accuracy: 0.783
ROC-AUC : 0.8584

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.75      0.76       661
           1       0.80      0.81      0.80       795

    accuracy                           0.78      1456
   macro avg       0.78      0.78      0.78      1456
weighted avg       0.78      0.78      0.78      1456



In [20]:
# Morgan 1024
X_train_t, X_test_t, y_train_t, y_test_t, X_train, X_test, y_train, y_test = prepare_data(
    df, fp_column="MorganFP_1024"
)

model_morgan1024, acc_morgan1024, roc_morgan1024, _, _ = train_and_evaluate(
    X_train_t, X_test_t, y_train_t, y_test_t,
    input_dim=X_train.shape[1],
    learning_rate=0.001,
    num_epochs=100,
    hidden1=256,
    hidden2=64,
    dropout=0.25,
    patience=15
)



Epoch [10/100] | Train Loss: 0.6220 | Test Loss: 0.6157
Epoch [20/100] | Train Loss: 0.4818 | Test Loss: 0.5127
Epoch [30/100] | Train Loss: 0.3966 | Test Loss: 0.5005
Epoch [40/100] | Train Loss: 0.3335 | Test Loss: 0.5114
Early stopping at epoch 42

Final Test Performance
Accuracy: 0.7768
ROC-AUC : 0.847

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.74      0.75       661
           1       0.79      0.81      0.80       795

    accuracy                           0.78      1456
   macro avg       0.78      0.77      0.77      1456
weighted avg       0.78      0.78      0.78      1456



In [21]:
# MACCS
X_train_t, X_test_t, y_train_t, y_test_t, X_train, X_test, y_train, y_test = prepare_data(
    df, fp_column="MACCSkeys"
)

model_maccs, acc_maccs, roc_maccs, _, _ = train_and_evaluate(
    X_train_t, X_test_t, y_train_t, y_test_t,
    input_dim=X_train.shape[1],
    learning_rate=0.001,
    num_epochs=100,
    hidden1=128,
    dropout=0.2,
    patience=15
)

Epoch [10/100] | Train Loss: 0.6372 | Test Loss: 0.6279
Epoch [20/100] | Train Loss: 0.5787 | Test Loss: 0.5758
Epoch [30/100] | Train Loss: 0.5412 | Test Loss: 0.5348
Epoch [40/100] | Train Loss: 0.5079 | Test Loss: 0.5034
Epoch [50/100] | Train Loss: 0.4752 | Test Loss: 0.4820
Epoch [60/100] | Train Loss: 0.4502 | Test Loss: 0.4655
Epoch [70/100] | Train Loss: 0.4167 | Test Loss: 0.4555
Epoch [80/100] | Train Loss: 0.3933 | Test Loss: 0.4492
Epoch [90/100] | Train Loss: 0.3611 | Test Loss: 0.4450
Epoch [100/100] | Train Loss: 0.3391 | Test Loss: 0.4392

Final Test Performance
Accuracy: 0.8036
ROC-AUC : 0.881

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.75      0.78       661
           1       0.80      0.85      0.82       795

    accuracy                           0.80      1456
   macro avg       0.80      0.80      0.80      1456
weighted avg       0.80      0.80      0.80      1456



In [22]:
results = pd.DataFrame({
    "Model": [
        "KNN (baseline)",
        "Decision Tree (baseline)",
        "Random Forest (baseline)",
        "Gradient Boosting (baseline)",
        "NN + MorganFP 1024",
        "NN + MorganFP 2048",
        "NN + MACCS"
    ],
    "Test Accuracy": [
        0.79,
        0.78,
        0.83,
        0.77,
        acc_morgan1024,
        acc_morgan2048,
        acc_maccs
    ],
    "Test ROC-AUC": [
        0.86,
        0.77,
        0.90,
        0.85,
        roc_morgan1024,
        roc_morgan2048,
        roc_maccs
    ]
})

results

,Model,Test Accuracy,Test ROC-AUC
0,KNN (baseline),0.790000,0.860000
1,Decision Tree (baseline),0.780000,0.770000
2,Random Forest (baseline),0.830000,0.900000
3,Gradient Boosting (baseline),0.770000,0.850000
4,NN + MorganFP 1024,0.776786,0.847033
5,NN + MorganFP 2048,0.782967,0.858409
6,NN + MACCS,0.803571,0.880977


7) Research how you can save the model / current state for later reuse. What are different options here? How can it be loaded again?

In [26]:
torch.save(model_morgan2048.state_dict(), "binclassifier_morgan2048_state.pth")

In [27]:
loaded_model = BinClassifierNN(input_dim=2048, hidden1=512, hidden2=128, dropout=0.3)
loaded_model.load_state_dict(torch.load("binclassifier_morgan2048_state.pth"))
loaded_model.eval()

BinClassifierNN(
  (net): Sequential(
    (0): Linear(in_features=2048, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [23]:
#Saving full model 
torch.save(model_morgan2048, "binclassifier_fullmodel.pth")


In [24]:
loaded_full_model = torch.load("binclassifier_fullmodel.pth")
loaded_full_model.eval()

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL __main__.BinClassifierNN was not an allowed global by default. Please use `torch.serialization.add_safe_globals([__main__.BinClassifierNN])` or the `torch.serialization.safe_globals([__main__.BinClassifierNN])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about type

#### 8) Discussion points
1) How did your model compare to other simple ML classifiers (all used the Morgan FPs)? Discuss!
2) Did you observe any difference between different fingerprint types?
3) Did the fingerprint size impact the model prediction? What message is to be learned from this?
4) What were some model parameters for decent performance depending on the fingerprint type? 
5) Was overfitting a problem? What approaches did you apply to limit that issue? What else would be possible
6) Consider the target "mutagenicity" in the context of molecular structure. What does noise mean here? How could you use such a predictive model in the lab? What other data-driven tools could be interesting in this QSAR context?
7) Why is exporting a full model usually not recommended?